# Reflection and Blogpost Writing

## Setup

In [4]:
import sys
print("Python:", sys.executable)

import autogen
print("AutoGen:", autogen.__version__)

import google.genai
print("google-genai: OK")

Python: C:\Users\gnana\anaconda3\envs\ai_project\python.exe
AutoGen: 0.7.2
google-genai: OK


In [5]:
from utils import get_openai_api_key
OPENAI_API_KEY = get_openai_api_key()
#print(OPENAI_API_KEY)
#llm_config = {"model": "gpt-3.5-turbo","api_key":OPENAI_API_KEY}
llm_config = {"model": "gemini-3.6-flash","api_key":"apikey
","api_type":"google"}
#llm_config = {"model": "llama3","api_type":"ollama"}


## The task!

In [6]:
task = '''
        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [7]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [8]:
reply = writer.generate_reply(messages=[{"content": task ,"role": "user"}])

C:\Users\gnana\anaconda3\envs\ai_project\Lib\site-packages\autogen\oai\gemini.py:803: UserWarning: Cost calculation is not implemented for model gemini-3.6-flash. Cost will be calculated zero.
  warnings.warn(


In [9]:
print(reply)

{'content': "# A Smarter Tomorrow: Why AI is Our Beacon of Hope\n\nFacing global crises, Artificial Intelligence is emerging as our greatest catalyst for good. \n\nAI is tackling humanity’s toughest puzzles. It’s discovering lifesaving drugs in record time, predicting climate shifts, and bridging educational divides. This isn't just about automation; it’s about empowerment. \n\nBy pairing human vision with machine intelligence, AI optimizes clean energy and democratizes knowledge worldwide. Technology alone won't save us, but combined with human empathy, AI offers the exact tools we need to build a better world. \n\nThe future isn't just digital—it's profoundly hopeful.", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [10]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [11]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

# AI with a Heart: Why "Hope AI" is Our Future

We often fear what artificial intelligence might take away, but Hope AI focuses on what it can give back. 

This emerging wave of compassionate tech targets humanity’s toughest challenges. From predicting climate disasters to democratizing global education and uncovering life-saving medical cures, Hope AI combines raw computing power with human empathy. 

It isn't about replacing humanity—it’s about amplifying our best intentions. Hope AI proves that the future of technology isn't just smart; it's deeply inspiring. 

The digital revolution finally has a heart. Are you ready to embrace it?

--------------------------------------------------------------------------------
Critic (to Writer):

H

## Nested chat

In [12]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [13]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [14]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [15]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [16]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [17]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [18]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

# AI with a Heart: Why "Hope AI" is Our Future

We often fear what artificial intelligence might take away, but Hope AI focuses on what it can give back. 

This emerging wave of compassionate tech targets humanity’s toughest challenges. From predicting climate disasters to democratizing global education and uncovering life-saving medical cures, Hope AI combines raw computing power with human empathy. 

It isn't about replacing humanity—it’s about amplifying our best intentions. Hope AI proves that the future of technology isn't just smart; it's deeply inspiring. 

The digital revolution finally has a heart. Are you ready to embrace it?

--------------------------------------------------------------------------------

*********************

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash
Please retry in 24.281130186s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.6-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 24
}
]

## Get the summary

In [19]:
print(res.summary)

# AI with a Heart: Why "Hope AI" is Our Future

We often fear what artificial intelligence might take away, but the "Hope AI" movement focuses on what it gives back.

This wave of compassionate tech tackles humanity’s greatest hurdles—from predicting climate disasters to uncovering life-saving medical cures. By combining raw computing power with human empathy, Hope AI isn't replacing humanity; it’s amplifying our best intentions.

The digital revolution finally has a heart. 

Where do you hope to see AI make the biggest impact? Drop your thoughts below!
